# 02 — Reparameterization

**Phase 2.** Put options on different strikes, days, and maturities onto comparable
axes — the step that lets a *surface* be treated as a *field*.

**Moneyness (standard deviations to expiry):**
$$z = \frac{\log(K/S)}{\sigma\,\sqrt{T}}$$

The $\sqrt{T}$ is **not optional**. Implied vol $\sigma$ is annualized, so without it
$z$ is not comparable across maturities and the whole $\tau$-dimension analysis is
distorted. (Equivalently, use total implied variance over the option's life.)

**Psychological time:** $\tau = \psi\,\log(1 + T/\psi)$ with $\psi \approx 20\text{–}30$
business days.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from src.data_pipeline import (
    compute_moneyness, compute_psychological_time, build_grid, ReparamConfig,
)
from src.plotting import apply_style
apply_style()

## 2.1 — Moneyness $z$: comparable across maturities (the $\sqrt{T}$ point)

In [ ]:
# A strike that is one standard deviation to expiry maps to z = 1 at ANY maturity.
S, sigma = 100.0, 0.2
for T in (0.25, 1.0, 2.0):
    K = S * np.exp(sigma * np.sqrt(T))            # exactly 1 s.d. to expiry
    z = compute_moneyness(K=K, S=S, sigma=sigma, T=T)
    print(f'T={T:>4}yr   K={K:7.2f}   z={z:.3f}')
print('\nWith the sqrt(T) scaling, all three give z = 1 (comparable). '
      'The log(K/S)/sigma shortcut would not.')

## 2.2 — Psychological time $\tau$ compresses long horizons

In [ ]:
ttoexp = np.linspace(1, 252, 200)   # business days, ~1 year
fig, ax = plt.subplots(figsize=(8, 5))
for psi in (15, 25, 40):
    ax.plot(ttoexp, compute_psychological_time(ttoexp, psi=psi), label=f'psi={psi}')
ax.set_xlabel('time to expiry (business days)'); ax.set_ylabel('tau')
ax.set_title('tau = psi log(1 + T/psi): short horizons spread out, long ones compressed')
ax.legend(); plt.tight_layout(); plt.show()

## 2.3 — Raw $(K, T)$ grid vs reparameterized $(z, \tau)$ grid — TODO

With real data loaded, bin options onto the `build_grid(...)` cells and plot the raw
strike/maturity grid next to the $(z,\tau)$ grid to show the transform visually.

In [ ]:
grid = build_grid(ReparamConfig(psi=25.0, n_z_bins=30, n_tau_bins=10))
print('z centers :', np.round(grid['z_centers'][:5], 2), '...')
print('tau centers:', np.round(grid['tau_centers'], 2))
# TODO (Phase 2): overlay real (K, ttoexp) samples mapped into (z, tau).

## 2.4 — Document your conventions

State these explicitly in the report — they change every downstream number:

- **Moneyness $\sigma$**: implied vol vs $\sqrt{\text{VSW}}$; annualized; the $\sqrt{T}$ scaling.
- **Hedging frequency**: daily / intraday — see `ReparamConfig.hedge_frequency`.
- **Maturity units**: business days for $\tau$; years for the $\sqrt{T}$ scaling (252 bd/yr).